In [1]:
import cv2
import xml.etree.ElementTree as ET
import os
import numpy as np
from IPython.display import Video, display

In [2]:
video_path = r"E:\DATA\Videos\NO20251114-152337-143741F.MP4"
xml_path   = r"E:\DATA\Annotations\NO20251114-152337-143741F.xml"
output_path = "annotated_demo.mp4"  # output video

In [3]:
tree = ET.parse(xml_path)
root = tree.getroot()

tracks = {}  # {track_id: [(frame, x, y)]}

for track in root.findall(".//track"):
    track_id = track.get("id")

    for box in track.findall("box"):
        frame = int(box.get("frame"))
        xtl = float(box.get("xtl"))
        ytl = float(box.get("ytl"))
        xbr = float(box.get("xbr"))
        ybr = float(box.get("ybr"))

        cx = (xtl + xbr) / 2
        cy = (ytl + ybr) / 2

        if track_id not in tracks:
            tracks[track_id] = []
        tracks[track_id].append((frame, cx, cy))

In [ ]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_index = 0

# Pre-build trajectory maps
trajectory_map = {tid: {} for tid in tracks}
for tid, points in tracks.items():
    for f, x, y in points:
        trajectory_map[tid][f] = (int(x), int(y))

# Start processing
while True:
    ret, frame = cap.read()
    if not ret:
        break

    for tid, frames in trajectory_map.items():
        # Draw trajectory up to this frame
        past_points = [(f, p) for f, p in frames.items() if f <= frame_index]
        past_points = [p for f, p in sorted(past_points)]

        for i in range(1, len(past_points)):
            cv2.line(frame, past_points[i-1], past_points[i], (0, 255, 0), 2)

    # Draw bounding box for this frame
    for tid, data in tracks.items():
        for f, cx, cy in data:
            if f == frame_index:
                # find full box from xml
                box_node = root.findall(f".//track[@id='{tid}']/box[@frame='{f}']")
                if box_node:
                    b = box_node[0]
                    xtl = int(float(b.get("xtl")))
                    ytl = int(float(b.get("ytl")))
                    xbr = int(float(b.get("xbr")))
                    ybr = int(float(b.get("ybr")))

                    cv2.rectangle(frame, (xtl, ytl), (xbr, ybr), (255, 0, 0), 2)
                    cv2.putText(frame, f"ID {tid}", (xtl, ytl - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    out.write(frame)
    frame_index += 1

cap.release()
out.release()

In [ ]:
display(Video("annotated_demo.mp4", embed=True))